# Frontend Generation from Python

This notebook explores how to generate HTML frontends directly from Python code — a powerful workflow for LLM-assisted development.

## Why Generate HTML from Python?

- **Rapid prototyping**: Skip the build tools, go straight to working HTML
- **Data-driven UIs**: Generate interfaces that match your data structure
- **LLM-friendly**: Single-language workflows are easier for AI to reason about
- **Portable artifacts**: Output is standalone HTML, no runtime dependencies

## Topics Covered

1. String formatting and f-strings
2. Jinja2 templates
3. IPython.display for inline previews
4. Building simple dashboards
5. Python frontend frameworks comparison
6. When to use Python vs React/Vue

## 1. Basic HTML Generation with F-Strings

The simplest approach: Python's f-strings for dynamic HTML.

In [ ]:
from IPython.display import HTML, display
import json

# Simple example: generate a styled card
def generate_card(title, content, color="#4A90E2"):
    html = f"""
    <div style="
        border: 2px solid {color};
        border-radius: 8px;
        padding: 20px;
        margin: 10px 0;
        background: linear-gradient(135deg, {color}22 0%, {color}11 100%);
    ">
        <h3 style="margin-top: 0; color: {color};">{title}</h3>
        <p style="margin-bottom: 0;">{content}</p>
    </div>
    """
    return html

# Display it inline
display(HTML(generate_card(
    "Welcome!",
    "This card was generated with a Python f-string. No templates, no frameworks.",
    "#E27D60"
)))

### Data-Driven UI Generation

Generate multiple components from data:

In [ ]:
# Sample data
metrics = [
    {"label": "Users", "value": "2,547", "change": "+12%", "color": "#41B883"},
    {"label": "Revenue", "value": "$45.2K", "change": "+8%", "color": "#42A5F5"},
    {"label": "Conversion", "value": "3.4%", "change": "-2%", "color": "#EF5350"},
]

def generate_metric_card(metric):
    return f"""
    <div style="
        display: inline-block;
        border: 1px solid #ddd;
        border-radius: 12px;
        padding: 20px;
        margin: 10px;
        min-width: 180px;
        text-align: center;
        box-shadow: 0 2px 8px rgba(0,0,0,0.1);
    ">
        <div style="color: #888; font-size: 14px; text-transform: uppercase;">{metric['label']}</div>
        <div style="font-size: 32px; font-weight: bold; color: {metric['color']}; margin: 10px 0;">{metric['value']}</div>
        <div style="color: {metric['color']}; font-weight: 600;">{metric['change']}</div>
    </div>
    """

# Generate dashboard
dashboard_html = "<div style='background: #f9f9f9; padding: 20px; border-radius: 8px;'>"
dashboard_html += "<h2 style='margin-top: 0;'>Dashboard Metrics</h2>"
for metric in metrics:
    dashboard_html += generate_metric_card(metric)
dashboard_html += "</div>"

display(HTML(dashboard_html))

## 2. Jinja2 Templates for Complex HTML

When HTML gets complex, templates are cleaner than f-strings.

In [ ]:
from jinja2 import Template

# Define a template
template_str = """
<!DOCTYPE html>
<html>
<head>
    <title>{{ title }}</title>
    <style>
        body { font-family: 'SF Pro', system-ui, sans-serif; padding: 20px; background: #f5f5f5; }
        .container { max-width: 800px; margin: 0 auto; background: white; padding: 30px; border-radius: 12px; }
        .task { padding: 15px; border-left: 4px solid {{ accent_color }}; margin: 10px 0; background: #fafafa; }
        .task.completed { opacity: 0.6; text-decoration: line-through; }
        h1 { color: {{ accent_color }}; }
    </style>
</head>
<body>
    <div class="container">
        <h1>{{ title }}</h1>
        <p>{{ description }}</p>
        
        {% for task in tasks %}
        <div class="task {% if task.completed %}completed{% endif %}">
            <strong>{{ task.name }}</strong> — {{ task.details }}
        </div>
        {% endfor %}
        
        <p style="margin-top: 30px; color: #888; font-size: 14px;">
            Generated at {{ timestamp }}
        </p>
    </div>
</body>
</html>
"""

# Render with data
template = Template(template_str)
html_output = template.render(
    title="Project Tasks",
    description="LLM-generated task dashboard",
    accent_color="#5D4FE3",
    tasks=[
        {"name": "Design mockup", "details": "Create Figma designs", "completed": True},
        {"name": "API integration", "details": "Connect to backend", "completed": False},
        {"name": "Testing", "details": "Write unit tests", "completed": False},
    ],
    timestamp="2026-02-10 23:30 PST"
)

# Preview in notebook
display(HTML(html_output))

# Save to file
with open('generated-dashboard.html', 'w') as f:
    f.write(html_output)
    
print("✅ Saved to generated-dashboard.html")

## 3. Interactive Components with Inline JavaScript

Add interactivity by embedding JavaScript in your generated HTML.

In [ ]:
def generate_interactive_form(fields):
    field_html = ""
    for field in fields:
        field_html += f"""
        <div style="margin-bottom: 15px;">
            <label style="display: block; margin-bottom: 5px; font-weight: 600;">{field['label']}</label>
            <input 
                type="{field['type']}" 
                id="{field['id']}" 
                placeholder="{field.get('placeholder', '')}" 
                style="width: 100%; padding: 10px; border: 1px solid #ddd; border-radius: 6px;"
            />
        </div>
        """
    
    html = f"""
    <div style="max-width: 500px; padding: 20px; border: 1px solid #ddd; border-radius: 12px; background: white;">
        <h3 style="margin-top: 0;">Dynamic Form</h3>
        {field_html}
        <button onclick="submitForm()" style="
            background: #5D4FE3; 
            color: white; 
            border: none; 
            padding: 12px 24px; 
            border-radius: 6px; 
            cursor: pointer;
            font-weight: 600;
        ">Submit</button>
        <div id="output" style="margin-top: 20px; padding: 10px; background: #f0f0f0; border-radius: 6px; display: none;"></div>
    </div>
    <script>
        function submitForm() {{
            const output = document.getElementById('output');
            const data = {{}};
            {chr(10).join([f"data['{field['id']}'] = document.getElementById('{field['id']}').value;" for field in fields])}
            output.innerHTML = '<strong>Submitted:</strong><br>' + JSON.stringify(data, null, 2);
            output.style.display = 'block';
        }}
    </script>
    """
    return html

form_fields = [
    {"id": "name", "label": "Name", "type": "text", "placeholder": "Enter your name"},
    {"id": "email", "label": "Email", "type": "email", "placeholder": "you@example.com"},
    {"id": "message", "label": "Message", "type": "text", "placeholder": "Your message"},
]

display(HTML(generate_interactive_form(form_fields)))

## 4. Chart Generation with Chart.js

Generate data visualizations by embedding Chart.js:

In [ ]:
def generate_chart(data, chart_type="bar", title="Chart"):
    chart_data = json.dumps(data)
    
    html = f"""
    <div style="max-width: 600px; padding: 20px; background: white; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
        <canvas id="myChart"></canvas>
    </div>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <script>
        const ctx = document.getElementById('myChart').getContext('2d');
        const chart = new Chart(ctx, {{
            type: '{chart_type}',
            data: {chart_data},
            options: {{
                responsive: true,
                plugins: {{
                    title: {{
                        display: true,
                        text: '{title}'
                    }}
                }}
            }}
        }});
    </script>
    """
    return html

chart_data = {
    "labels": ["Jan", "Feb", "Mar", "Apr", "May"],
    "datasets": [{
        "label": "Revenue",
        "data": [12, 19, 8, 15, 22],
        "backgroundColor": "rgba(93, 79, 227, 0.5)",
        "borderColor": "rgba(93, 79, 227, 1)",
        "borderWidth": 2
    }]
}

display(HTML(generate_chart(chart_data, chart_type="line", title="Monthly Revenue")))

## 5. Python Frontend Frameworks Comparison

When Python-only frontends make sense:

### Streamlit
```python
import streamlit as st

st.title("My App")
name = st.text_input("Name")
st.write(f"Hello, {name}!")
```

**Pros:**
- Zero frontend code
- Great for data science demos
- Built-in widgets and charts

**Cons:**
- Limited customization
- Requires running server
- Reloads on every interaction

**Use when:** Building internal data tools, ML demos, quick prototypes

---

### Gradio
```python
import gradio as gr

def greet(name):
    return f"Hello {name}!"

gr.Interface(fn=greet, inputs="text", outputs="text").launch()
```

**Pros:**
- Perfect for ML model interfaces
- Auto-generates API
- Shareable links

**Cons:**
- Less flexible than Streamlit
- Best for single-purpose UIs

**Use when:** Demoing ML models, creating quick API frontends

---

### NiceGUI
```python
from nicegui import ui

@ui.page('/')
def index():
    ui.label('Hello NiceGUI!')
    ui.button('Click me', on_click=lambda: ui.notify('Clicked!'))

ui.run()
```

**Pros:**
- Modern, reactive UI
- More control than Streamlit
- Built on Vue.js under the hood

**Cons:**
- Newer, smaller ecosystem
- Still requires running server

**Use when:** You want more control but stay in Python

---

### Reflex (formerly Pynecone)
```python
import reflex as rx

def index():
    return rx.text("Hello Reflex!")

app = rx.App()
app.add_page(index)
```

**Pros:**
- Full-stack Python framework
- Compiles to React
- Production-ready apps

**Cons:**
- Steeper learning curve
- More complex setup

**Use when:** Building full production apps in Python

---

### Static HTML Generation (This Notebook's Approach)

**Pros:**
- No server required (just open in browser)
- Completely portable
- Full control over output
- Easy to version control
- Works with any hosting (S3, GitHub Pages, etc.)

**Cons:**
- No backend logic after generation
- Need to regenerate for updates
- Limited to client-side interactivity

**Use when:**
- Generating reports or dashboards
- Creating artifacts from data
- Building prototypes for review
- Making standalone tools
- LLM-assisted frontend generation

## 6. When to Use Python vs React/Vue

### Choose Python-generated HTML when:

✅ **Static content from data** — Reports, dashboards, documentation  
✅ **Rapid prototyping** — Get something visual fast  
✅ **Data science workflows** — Jupyter → HTML artifacts  
✅ **LLM-assisted development** — Single language is easier to prompt  
✅ **Internal tools** — Not customer-facing, just needs to work  
✅ **No backend needed** — Pure client-side interactivity  

### Choose React/Vue when:

✅ **Complex interactivity** — Real-time updates, state management  
✅ **Team collaboration** — Frontend specialists involved  
✅ **Production apps** — Customer-facing, needs polish  
✅ **Rich ecosystem needs** — Component libraries, routing, etc.  
✅ **Mobile responsiveness** — Complex responsive layouts  
✅ **Real-time data** — WebSocket connections, live updates  

### Hybrid Approach: Python → React Artifact

Use Python to generate:
- Initial React component structure
- Mock data and API interfaces
- Configuration files
- Boilerplate code

Then hand off to frontend tools for refinement.

## 7. Complete Example: LLM-Generated Dashboard

Putting it all together — a full dashboard generator:

In [ ]:
from jinja2 import Template
from datetime import datetime

dashboard_template = Template("""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{{ title }}</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { font-family: -apple-system, system-ui, sans-serif; background: #f5f7fa; }
        .header { background: linear-gradient(135deg, {{ theme.primary }} 0%, {{ theme.secondary }} 100%); color: white; padding: 30px; }
        .container { max-width: 1200px; margin: 0 auto; padding: 20px; }
        .metrics { display: grid; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr)); gap: 20px; margin: 20px 0; }
        .metric-card { background: white; padding: 20px; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); }
        .metric-value { font-size: 36px; font-weight: bold; color: {{ theme.primary }}; margin: 10px 0; }
        .metric-label { color: #888; text-transform: uppercase; font-size: 12px; letter-spacing: 1px; }
        .chart-container { background: white; padding: 20px; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); margin: 20px 0; }
        .footer { text-align: center; padding: 20px; color: #888; font-size: 14px; }
    </style>
</head>
<body>
    <div class="header">
        <div class="container">
            <h1>{{ title }}</h1>
            <p>{{ subtitle }}</p>
        </div>
    </div>
    
    <div class="container">
        <div class="metrics">
            {% for metric in metrics %}
            <div class="metric-card">
                <div class="metric-label">{{ metric.label }}</div>
                <div class="metric-value">{{ metric.value }}</div>
                <div style="color: {{ metric.trend_color }}; font-weight: 600;">{{ metric.change }}</div>
            </div>
            {% endfor %}
        </div>
        
        <div class="chart-container">
            <canvas id="mainChart"></canvas>
        </div>
    </div>
    
    <div class="footer">
        Generated at {{ timestamp }} | Made with Python 🐍
    </div>
    
    <script>
        const ctx = document.getElementById('mainChart').getContext('2d');
        new Chart(ctx, {
            type: 'line',
            data: {{ chart_data | tojson }},
            options: {
                responsive: true,
                plugins: {
                    legend: { position: 'top' },
                    title: { display: true, text: '{{ chart_title }}' }
                },
                scales: {
                    y: { beginAtZero: true }
                }
            }
        });
    </script>
</body>
</html>
""")

# Generate a complete dashboard
dashboard_html = dashboard_template.render(
    title="Analytics Dashboard",
    subtitle="Real-time business metrics",
    theme={
        "primary": "#5D4FE3",
        "secondary": "#8B7FE8"
    },
    metrics=[
        {"label": "Total Users", "value": "12,547", "change": "+12% vs last month", "trend_color": "#10B981"},
        {"label": "Revenue", "value": "$87.5K", "change": "+8% vs last month", "trend_color": "#10B981"},
        {"label": "Conversion", "value": "3.8%", "change": "-2% vs last month", "trend_color": "#EF4444"},
        {"label": "Avg. Session", "value": "4m 32s", "change": "+15% vs last month", "trend_color": "#10B981"},
    ],
    chart_data={
        "labels": ["Jan", "Feb", "Mar", "Apr", "May", "Jun"],
        "datasets": [
            {
                "label": "Users",
                "data": [1200, 1900, 1500, 2200, 2800, 3100],
                "borderColor": "#5D4FE3",
                "backgroundColor": "rgba(93, 79, 227, 0.1)",
                "tension": 0.4
            },
            {
                "label": "Revenue",
                "data": [3000, 5000, 4000, 6000, 7500, 8750],
                "borderColor": "#10B981",
                "backgroundColor": "rgba(16, 185, 129, 0.1)",
                "tension": 0.4
            }
        ]
    },
    chart_title="Growth Trends",
    timestamp=datetime.now().strftime("%Y-%m-%d %H:%M:%S")
)

# Save to file
with open('complete-dashboard.html', 'w') as f:
    f.write(dashboard_html)

print("✅ Dashboard saved to complete-dashboard.html")
print("📊 Open it in your browser to see the full interactive dashboard")

# Preview a snippet in the notebook
display(HTML("""
<div style="padding: 20px; background: #e3f2fd; border-left: 4px solid #2196F3; border-radius: 4px;">
    <strong>✅ Success!</strong> Your dashboard has been generated.<br>
    <em>Open <code>complete-dashboard.html</code> in a browser to view the full version.</em>
</div>
"""))

## Key Takeaways

1. **Python can generate production-ready HTML** — No frontend framework required
2. **Jinja2 > f-strings for complex templates** — Separation of concerns matters
3. **IPython.display.HTML() is your friend** — Instant feedback loop
4. **Static HTML is underrated** — Portable, fast, version-controllable
5. **Know when to stop** — Python frontends have limits; React/Vue exist for a reason

## Next Steps

- Try the `ui-gen.sh` script to automate this workflow
- Experiment with `serve-and-watch.sh` for live development
- Use `screenshot-diff.sh` to compare iterations
- Prompt an LLM to generate dashboard components
- Build a personal dashboard from your own data

## Resources

- [Jinja2 Documentation](https://jinja.palletsprojects.com/)
- [Chart.js](https://www.chartjs.org/)
- [Streamlit](https://streamlit.io/)
- [Gradio](https://gradio.app/)
- [NiceGUI](https://nicegui.io/)
- [Reflex](https://reflex.dev/)